In [ ]:
import time
from google import genai

client = genai.Client()

job_name = "batches/d2693089rs502ku7ehajsjejnoyxxbgabzdg"
batch_job = client.batches.get(name=job_name)

completed_states = set([
    'JOB_STATE_SUCCEEDED',
    'JOB_STATE_FAILED',
    'JOB_STATE_CANCELLED',
    'JOB_STATE_EXPIRED',
])

In [ ]:
print(f"Polling status for job: {job_name}")
batch_job = client.batches.get(name=job_name) # Initial get
print(f"Current state: {batch_job.state.name}")

Polling status for job: batches/d2693089rs502ku7ehajsjejnoyxxbgabzdg
Current state: JOB_STATE_SUCCEEDED


In [16]:
import os

if batch_job.state.name == 'JOB_STATE_SUCCEEDED':
    print(batch_job.dest)
    print(batch_job.dest.file_name)
    
    result_file_name = batch_job.dest.file_name
    file_content = client.files.download(file=result_file_name)
    
    out_file_name = "results/text_image_concept_collapse/data/text_generations.json"
    os.makedirs(
        os.path.dirname(out_file_name), 
        exist_ok=True
    )
    
    with open(out_file_name, 'w') as f:
        f.write(file_content.decode('utf-8'))
    

format=None gcs_uri=None bigquery_uri=None file_name='files/batch-d2693089rs502ku7ehajsjejnoyxxbgabzdg' inlined_responses=None inlined_embed_content_responses=None vertex_dataset=None
files/batch-d2693089rs502ku7ehajsjejnoyxxbgabzdg


In [37]:
import json
in_file_name = "experiments/text_image_concept_collapse/text_response_batch.jsonl"
with open(out_file_name) as f:
    responses = [json.loads(line) for line in f]

with open(in_file_name) as f:
    prompts = [json.loads(line) for line in f]

print(json.dumps(responses[0], indent=4))

{
    "key": "req-0",
    "response": {
        "candidates": [
            {
                "content": {
                    "parts": [
                        {
                            "text": "**My Thought Process: Crafting a Nature Scene**\n\nOkay, so the prompt is straightforward enough. It\u2019s a nature scene, single paragraph, direct response - no problem. Time to think about how I want to approach this, in a concise way.\n\nFirst, let's see, what imagery feels right? A quiet forest morning is immediately appealing. Mist, sunrise, and a creek are solid starting points. I want to build on that. Now, I need sensory details. Sight: golden light, emerald canopies, glistening dew. Sound: babbling brook, birdsong, leaves rustling. Smell/Feel: Crisp pine air, cool dampness. What kind of action will draw the reader in? Let's go with shifting light, maybe a deer or bird, or just the natural flow of the water.\n\nNow to formulate it all into a paragraph. I can envision the scene \u

In [40]:
def create_row(p, r):
    assert(p['key'] == r['key'])
    thoughts = []
    response = ''
    for t in r['response']['candidates'][0]['content']['parts']:
        if 'thought' in t and t['thought']:
            thoughts.append(t['text'])
        if not 'thought' in t or not t['thought']:
            response = t['text']
    if len(thoughts) != 1:
        print(f'Thought lenght was actually {len(thoughts)}')
    return {
        "query": p['request']['contents'][0]['parts'][0]['text'],
        "system_instruction": p['request']['system_instruction']['parts'][0]['text'],
        "thinking_level": p['request']['generationConfig']['thinkingConfig']['thinking_level'],
        "response": response,
        "thoughts": thoughts,
        "model": r['response']['modelVersion']
    }

create_row(prompts[0], responses[0])

{'query': 'Write a paragraph on a scene in nature.',
 'system_instruction': 'Respond to the prompt directly.',
 'thinking_level': 'MEDIUM',
 'response': 'Golden morning light filtered through the towering canopy of ancient pines, fracturing into soft, luminous beams that illuminated the damp forest floor below. A gentle mist clung low to a winding, crystal-clear stream, curling like smoke over moss-carpeted stones worn smooth by centuries of patient water. Dewdrops hung suspended from the delicate fronds of vibrant ferns, catching the early sun and glittering like scattered diamonds against the rich, dark earth. The air was cool and invigorating, saturated with the earthy scent of fallen leaves and pine needles, while the only sounds breaking the profound stillness were the rhythmic murmur of the brook and the melodic, tentative trill of an unseen songbird greeting the dawn.',
 'thoughts': ["**My Thought Process: Crafting a Nature Scene**\n\nOkay, so the prompt is straightforward enoug

In [43]:
rows = []
for prompt, response in zip(prompts, responses):
    rows.append(create_row(prompt, response))

import pandas as pd
df = pd.DataFrame(rows)

df.to_csv("results/text_image_concept_collapse/data/text_experiment.csv")

Thought lenght was actually 0
